# Deciphering Gene Regulatory Mechanisms in Leukemic and Immune Cells
### A Python single-cell gene regulatory network (GRN) workflow using public Nature Portfolio benchmark datasets

**Primary runnable benchmark:** **GSE211036**, single-cell RNA-seq from leukemic stem-cell and blast populations in CEBPA-double-mutant acute myeloid leukemia (AML), reported by Adamo et al. in *Leukemia* (Springer Nature / Nature Portfolio).

**Cross-context benchmark for leukemic versus normal hematopoietic / immune states:** **GSE139369**, the *Nature Biotechnology* study of single-cell multi-omic profiles from healthy blood/bone marrow and mixed-phenotype acute leukemia (MPAL). Its released transcriptome files are RDS-format; use a processed AnnData (`.h5ad`) version in the optional cross-dataset section, or convert the RDS counts to AnnData before applying the same pipeline.

**Additional AML context:** **GSE199336**, associated with *Nature Medicine*, provides data for studying AML cellular hierarchy and drug response.

## What this notebook does
1. Downloads and inspects the public GSE211036 supplementary archive.
2. Loads a compatible single-cell expression matrix (10x HDF5, Matrix Market, H5AD, or a tabular counts matrix).
3. Performs basic QC, normalization, exploratory embedding and clustering.
4. Infers a **candidate TF→target network** using tree-ensemble feature importance (a GENIE3-like approach).
5. Exports edge tables and visualizations, and summarizes regulator expression across inferred clusters.
6. Describes how to reuse the pipeline with GSE139369 / GSE199336 to assess shared versus context-specific associations.

> **Important interpretation:** GSE211036 profiles one CEBPA-double-mutant AML sample with LSC and blast populations; it is not by itself a normal immune-cell comparison. Expression-derived links are *candidate predictive associations*, not proof of direct TF binding or causal regulation. For stronger evidence, integrate matched chromatin accessibility, TF motif, ChIP/CUT&Tag, promoter-capture Hi-C, perturbation and multi-patient evidence.


## 1. Install dependencies

Run once in the notebook environment. A restart may be needed if packages were already imported in the kernel.

In [ ]:
%pip -q install scanpy anndata scikit-learn networkx matplotlib pandas numpy scipy requests igraph leidenalg

In [ ]:
from pathlib import Path
import os, tarfile, gzip, shutil, requests, warnings, re
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.cluster import KMeans
from IPython.display import display

SEED = 17
np.random.seed(SEED)
sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, figsize=(6, 5), facecolor="white")

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "GSE211036"
RESULTS_DIR = Path("results")
EXTERNAL_DIR = DATA_DIR / "external"
for directory in (DATA_DIR, RAW_DIR, RESULTS_DIR, EXTERNAL_DIR):
    directory.mkdir(parents=True, exist_ok=True)

GSE211036_URL = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE211nnn/GSE211036/suppl/GSE211036_RAW.tar"
print("Scanpy:", sc.__version__)
print("Working directory:", Path.cwd())

## 2. Download and inspect GSE211036

The GEO supplementary archive is approximately 15 MB according to GEO at the time this notebook was prepared. If an institution or network blocks direct download, download `GSE211036_RAW.tar` from [NCBI GEO](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE211036) in a browser and save it to `data/GSE211036/`.

In [ ]:
archive_path = RAW_DIR / "GSE211036_RAW.tar"

if not archive_path.exists():
    try:
        with requests.get(GSE211036_URL, stream=True, timeout=60) as response:
            response.raise_for_status()
            with open(archive_path, "wb") as handle:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        handle.write(chunk)
        print(f"Downloaded {archive_path} ({archive_path.stat().st_size / 1e6:.1f} MB)")
    except Exception as exc:
        print("Automatic download failed:", repr(exc))
        print("Download the RAW archive manually from the GEO link in the preceding markdown cell,")
        print("then place it here:", archive_path.resolve())

if archive_path.exists():
    # Extract safely: reject absolute paths and parent traversal entries.
    with tarfile.open(archive_path, "r:*") as tar:
        members = []
        for member in tar.getmembers():
            member_path = Path(member.name)
            if member_path.is_absolute() or ".." in member_path.parts:
                continue
            members.append(member)
        tar.extractall(path=RAW_DIR, members=members)
    print("Archive extracted to:", RAW_DIR.resolve())
else:
    print("No archive present yet; subsequent loading cell will report the discovered files.")

all_files = sorted(p for p in RAW_DIR.rglob("*") if p.is_file())
print(f"Files under {RAW_DIR}: {len(all_files)}")
for path in all_files[:80]:
    print(f"{path.relative_to(RAW_DIR)}  ({path.stat().st_size / 1024:.1f} KB)")
if len(all_files) > 80:
    print("... file listing truncated")

## 3. Load single-cell expression data

The loader tries H5AD, 10x HDF5, Matrix Market, then a tabular matrix. It reports a clear error if the GEO archive uses a format that needs custom parsing. For GEO matrices stored as **genes × cells**, the loader transposes them to **cells × genes**, as expected by AnnData.

In [ ]:
def _read_tabular_matrix(path: Path):
    """Best-effort reader for a dense genes-by-cells or cells-by-genes TSV/CSV matrix."""
    try:
        sep = "\t" if path.name.lower().endswith((".tsv", ".txt", ".tsv.gz", ".txt.gz")) else ","
        df = pd.read_csv(path, sep=sep, index_col=0, compression="infer", low_memory=False)
        if df.shape[0] < 30 or df.shape[1] < 5:
            return None
        # Only accept tables that are overwhelmingly numeric after the first/index column.
        numeric = df.apply(pd.to_numeric, errors="coerce")
        numeric_fraction = float(numeric.notna().to_numpy().mean())
        if numeric_fraction < 0.90:
            return None
        numeric = numeric.fillna(0)
        # Expression matrices conventionally have many more genes than cells.
        if numeric.shape[0] > numeric.shape[1]:
            genes = numeric.index.astype(str)
            cells = numeric.columns.astype(str)
            matrix = sp.csr_matrix(numeric.to_numpy(dtype=np.float32).T)
        else:
            cells = numeric.index.astype(str)
            genes = numeric.columns.astype(str)
            matrix = sp.csr_matrix(numeric.to_numpy(dtype=np.float32))
        obs = pd.DataFrame(index=pd.Index(cells, name="cell_id"))
        var = pd.DataFrame(index=pd.Index(genes, name="gene"))
        result = ad.AnnData(X=matrix, obs=obs, var=var)
        result.var_names_make_unique()
        return result
    except Exception:
        return None


def load_expression_data(root: Path):
    """Try common single-cell formats recursively and return AnnData."""
    files = [p for p in root.rglob("*") if p.is_file()]
    if not files:
        raise FileNotFoundError(
            f"No expression files found in {root}. Download GSE211036_RAW.tar from GEO first."
        )

    # Explicit H5AD first.
    for path in files:
        if path.suffix.lower() == ".h5ad":
            try:
                print("Reading H5AD:", path)
                return sc.read_h5ad(path)
            except Exception as exc:
                print("Could not read", path.name, "as H5AD:", str(exc)[:180])

    # Standard 10x HDF5.
    for path in files:
        if path.suffix.lower() in {".h5", ".hdf5"}:
            try:
                print("Trying 10x HDF5:", path)
                matrix = sc.read_10x_h5(path, gex_only=True)
                matrix.var_names_make_unique()
                return matrix
            except Exception as exc:
                print("Could not read", path.name, "as 10x HDF5:", str(exc)[:180])

    # Matrix Market directories (matrix.mtx[.gz] + features/genes + barcodes).
    for path in files:
        if path.name.lower() in {"matrix.mtx", "matrix.mtx.gz"}:
            try:
                print("Trying 10x Matrix Market directory:", path.parent)
                try:
                    matrix = sc.read_10x_mtx(path.parent, var_names="gene_symbols", make_unique=True)
                except Exception:
                    matrix = sc.read_10x_mtx(path.parent, var_names="gene_ids", make_unique=True)
                return matrix
            except Exception as exc:
                print("Could not read Matrix Market folder", path.parent, ":", str(exc)[:180])

    # Tabular matrices. Avoid obvious metadata / annotation tables.
    tabular_suffixes = (".tsv", ".txt", ".csv", ".tsv.gz", ".csv.gz", ".txt.gz")
    candidates = [p for p in files if p.name.lower().endswith(tabular_suffixes)]
    candidates.sort(key=lambda p: p.stat().st_size, reverse=True)
    for path in candidates:
        if path.stat().st_size < 20_000:
            continue
        matrix = _read_tabular_matrix(path)
        if matrix is not None:
            print("Loaded tabular expression matrix:", path)
            print("Interpreted shape (cells, genes):", matrix.shape)
            return matrix

    examples = "\n".join(str(p.relative_to(root)) for p in files[:50])
    raise RuntimeError(
        "Could not automatically identify a supported expression matrix in the downloaded archive. "
        "The loader supports .h5ad, 10x .h5, Matrix Market folders, and dense TSV/CSV matrices. "
        "Inspect the file list below and adapt the loader if necessary.\n\nFiles found:\n" + examples
    )

adata = load_expression_data(RAW_DIR)
print(adata)
print("First gene names:", list(adata.var_names[:10]))
print("First cell IDs:", list(adata.obs_names[:5]))

## 4. Quality control and normalization

The default rules are intentionally conservative and should be reviewed for each sample. Keep an untouched raw-count layer for later re-analysis. For the primary GEO archive the expected input is raw scRNA-seq counts; if your input is already normalized, the code detects this heuristically and avoids normalizing it twice.

In [ ]:
# Use sparse storage where possible and remove empty/duplicate labels.
adata.var_names = adata.var_names.astype(str)
adata.obs_names = adata.obs_names.astype(str)
adata.var_names_make_unique()

# Save an untouched copy of the expression values before filtering.
if "counts" not in adata.layers:
    adata.layers["counts"] = adata.X.copy()

# Basic QC metrics. Calculate mitochondrial percentage only if human-style MT genes are present.
adata.var["mt"] = adata.var_names.str.upper().str.startswith(("MT-", "MT."))
try:
    sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True, percent_top=None, log1p=False)
except Exception:
    sc.pp.calculate_qc_metrics(adata, inplace=True, percent_top=None, log1p=False)

qc_columns = [c for c in ["n_genes_by_counts", "total_counts", "pct_counts_mt"] if c in adata.obs.columns]
if qc_columns:
    display(adata.obs[qc_columns].describe().T)

# Plot available QC metrics.
plot_cols = [c for c in ["n_genes_by_counts", "total_counts", "pct_counts_mt"] if c in adata.obs.columns]
if plot_cols:
    ncols = min(3, len(plot_cols))
    fig, axes = plt.subplots(1, ncols, figsize=(4 * ncols, 3))
    axes = np.atleast_1d(axes)
    for ax, col in zip(axes, plot_cols):
        ax.hist(adata.obs[col].dropna(), bins=40)
        ax.set_title(col)
        ax.set_xlabel(col)
    plt.tight_layout()
    plt.show()

# Apply a minimum genes-per-cell threshold scaled down for small matrices.
min_genes_threshold = min(100, max(10, int(adata.n_vars * 0.005)))
sc.pp.filter_cells(adata, min_genes=min_genes_threshold)
sc.pp.filter_genes(adata, min_cells=min(3, max(1, adata.n_obs // 10)))
if "pct_counts_mt" in adata.obs.columns and bool(adata.var["mt"].any()):
    # A broad 25% threshold avoids excluding most cells from disease samples; review for your data.
    adata = adata[adata.obs["pct_counts_mt"].fillna(0) < 25, :].copy()

# Heuristic: treat near-integer values with a high maximum as raw counts; otherwise assume transformed input.
Xsample = adata.X.data if sp.issparse(adata.X) else np.asarray(adata.X).ravel()
if Xsample.size > 100_000:
    Xsample = Xsample[np.random.choice(Xsample.size, 100_000, replace=False)]
finite_sample = Xsample[np.isfinite(Xsample)]
max_value = float(np.max(finite_sample)) if finite_sample.size else 0.0
integer_like = bool(finite_sample.size and np.allclose(finite_sample[:min(10000, len(finite_sample))], np.round(finite_sample[:min(10000, len(finite_sample))]), atol=1e-6))

if integer_like and max_value > 20:
    print("Input appears to be raw counts: normalize_total + log1p will be applied.")
    adata.layers["counts"] = adata.X.copy()
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
else:
    print("Input does not look like raw integer counts; assuming values are already normalized/log-transformed.")

print("Post-QC AnnData:", adata)
print("Matrix max after preprocessing:", float(adata.X.max()) if sp.issparse(adata.X) else float(np.max(adata.X)))

## 5. Explore cell states

The dataset metadata describe sorted leukemic stem-cell (LSC) and blast populations. If cell-level labels are not included in the expression archive, unsupervised clusters are **computational clusters**, not definitive cell-type labels. The UMAP and cluster labels are exploratory.

In [ ]:
# Highly variable genes for embedding; use all genes if the matrix is small.
if adata.n_vars >= 30:
    n_hvg = min(2000, adata.n_vars - 1)
    try:
        sc.pp.highly_variable_genes(adata, n_top_genes=n_hvg, flavor="seurat")
    except Exception as exc:
        print("HVG selection fallback:", exc)
        adata.var["highly_variable"] = True
else:
    adata.var["highly_variable"] = True

n_pcs = min(30, max(2, adata.n_obs - 1), max(2, int(adata.var["highly_variable"].sum()) - 1))
if adata.n_obs >= 10 and adata.n_vars >= 10:
    sc.tl.pca(adata, n_comps=n_pcs, use_highly_variable=True, svd_solver="arpack")
    sc.pp.neighbors(adata, n_neighbors=min(15, max(2, adata.n_obs - 1)), n_pcs=n_pcs)
    sc.tl.umap(adata, random_state=SEED)
    try:
        sc.tl.leiden(adata, resolution=0.6, key_added="grn_cluster", random_state=SEED)
    except Exception as exc:
        print("Leiden clustering unavailable; using K-means on PCA coordinates:", str(exc)[:200])
        n_clusters = min(5, max(2, int(np.sqrt(adata.n_obs / 10))))
        adata.obs["grn_cluster"] = KMeans(n_clusters=n_clusters, random_state=SEED, n_init=10).fit_predict(adata.obsm["X_pca"]).astype(str)
    sc.pl.umap(adata, color="grn_cluster", title="GSE211036: exploratory cell clusters")
    display(adata.obs["grn_cluster"].value_counts().rename("n_cells").to_frame())
else:
    print("Too few cells/genes for a stable PCA/UMAP; clustering skipped.")
    adata.obs["grn_cluster"] = "all_cells"

# Save a processed copy to make the workflow reproducible.
processed_path = RESULTS_DIR / "GSE211036_preprocessed.h5ad"
adata.write_h5ad(processed_path)
print("Saved:", processed_path.resolve())

## 6. Define candidate transcription factors (TFs)

This is a **curated starter panel**, not a complete human TF catalogue. It emphasizes myeloid/AML regulators and immune-lineage factors. Only genes found in the expression matrix and sufficiently variable across cells are used. For publication-quality inference, replace/extend this list with a versioned human TF catalogue (for example, AnimalTFDB or TF checkpoint from DoRothEA) and document the version.

In [ ]:
TF_CANDIDATES = [
    # Myeloid / AML core and lineage factors
    "CEBPA", "CEBPB", "CEBPD", "CEBPE", "RUNX1", "RUNX2", "RUNX3", "SPI1", "GATA1", "GATA2",
    "TAL1", "LMO2", "ERG", "FLI1", "MYB", "MYC", "MEF2C", "GFI1", "GFI1B", "KLF4", "KLF6",
    "IRF8", "IRF1", "IRF2", "IRF4", "IRF7", "IRF9", "STAT1", "STAT3", "STAT5A", "STAT5B",
    # AP-1, stress and inflammatory factors
    "JUN", "JUNB", "JUND", "FOS", "FOSB", "FOSL1", "FOSL2", "ATF3", "BATF", "BATF3",
    "NFKB1", "RELA", "REL", "BCL3", "BACH1", "BACH2", "CEBPZ",
    # Lymphoid / hematopoietic / general regulatory factors
    "PAX5", "TCF3", "TCF7", "LEF1", "IKZF1", "IKZF2", "IKZF3", "BCL11A", "E2F1", "FOXO1",
    "FOXO3", "YY1", "EZH2", "ZEB2", "RUNX1T1", "NR4A1", "NR4A2", "NR4A3", "KLF2", "KLF10",
    "HIF1A", "PPARG", "CEBPG", "ELF1", "ETS1", "ETV6", "GABPA", "MAF", "MAFB", "MITF"
]

present_genes = set(adata.var_names)
expressed_tf_candidates = [g for g in TF_CANDIDATES if g in present_genes]
print(f"Curated TFs present: {len(expressed_tf_candidates)} / {len(TF_CANDIDATES)}")
print(expressed_tf_candidates)

if len(expressed_tf_candidates) < 3:
    raise ValueError(
        "Fewer than 3 curated TF symbols are present. Check whether the matrix uses Ensembl IDs rather than gene symbols. "
        "Map Ensembl IDs to approved HGNC symbols before GRN inference."
    )

## 7. Infer a candidate TF→target network

For each selected target gene, an `ExtraTreesRegressor` predicts its expression from the expression of available TF candidates. Feature importance ranks TFs that help predict each target. This is computationally accessible for a small/moderate target panel and analogous in spirit to GENIE3; it is **not** a full implementation of GENIE3 and does not establish causality.

The default below uses up to 250 target genes and at most 1,500 cells for speed. Increase these after checking runtime and memory. You may also set a larger `MAX_TARGETS`, or use GENIE3/GRNBoost2/SCENIC for a more formal benchmark.

In [ ]:
MAX_CELLS_FOR_GRN = 1500
MAX_TARGETS = 250
MAX_TFS = 60
N_TREES = 60
TOP_REGULATORS_PER_TARGET = 5
MIN_EDGE_IMPORTANCE = 0.0  # set >0 to make the network more stringent

# Restrict to TFs with measurable expression and variance; do not use a constant predictor.
expr_all = adata.X
if sp.issparse(expr_all):
    gene_means = np.asarray(expr_all.mean(axis=0)).ravel()
    gene_sq_means = np.asarray(expr_all.power(2).mean(axis=0)).ravel()
    gene_vars = np.maximum(gene_sq_means - gene_means**2, 0)
else:
    gene_means = np.asarray(expr_all).mean(axis=0)
    gene_vars = np.asarray(expr_all).var(axis=0)

var_lookup = {g: i for i, g in enumerate(adata.var_names)}
regulators = [g for g in expressed_tf_candidates if gene_vars[var_lookup[g]] > 1e-10 and gene_means[var_lookup[g]] > 0.01]
regulators = regulators[:MAX_TFS]
if len(regulators) < 3:
    raise ValueError("Too few variable/expression-supported TFs remain after filtering. Adjust TF panel or QC.")

# Target candidates: curated phenotype/lineage genes first if present, then highly variable genes.
TARGET_PRIORITY = [
    "CEBPA", "CEBPB", "RUNX1", "SPI1", "CD69", "MPO", "LYZ", "ITGAM", "S100A8", "S100A9",
    "FCGR3B", "LST1", "CTSS", "HLA-DRA", "CD74", "NKG7", "CD3D", "CD3E", "MS4A1", "CD79A",
    "IRF8", "BATF", "JUN", "FOS", "MKI67", "PROM1", "CD34", "GATA2", "MEF2C"
]
priority_targets = [g for g in TARGET_PRIORITY if g in var_lookup and gene_vars[var_lookup[g]] > 1e-10]
if "highly_variable" in adata.var.columns:
    hvg_genes = list(adata.var_names[adata.var["highly_variable"].fillna(False)])
else:
    hvg_genes = list(adata.var_names[np.argsort(gene_vars)[::-1][:min(2000, adata.n_vars)]])
hvg_genes = sorted(hvg_genes, key=lambda g: gene_vars[var_lookup[g]], reverse=True)
all_targets = list(dict.fromkeys(priority_targets + hvg_genes))
all_targets = [g for g in all_targets if gene_vars[var_lookup[g]] > 1e-10]
target_genes = all_targets[:MAX_TARGETS]

# Subsample cells if needed. Subsample without replacement and preserve all selected groups as best effort.
if adata.n_obs > MAX_CELLS_FOR_GRN:
    cell_indices = np.random.default_rng(SEED).choice(adata.n_obs, MAX_CELLS_FOR_GRN, replace=False)
else:
    cell_indices = np.arange(adata.n_obs)

reg_idx = [var_lookup[g] for g in regulators]
target_idx = [var_lookup[g] for g in target_genes]
X_tf = adata.X[cell_indices, reg_idx]
X_targets = adata.X[cell_indices, target_idx]
X_tf = X_tf.toarray() if sp.issparse(X_tf) else np.asarray(X_tf)
X_targets = X_targets.toarray() if sp.issparse(X_targets) else np.asarray(X_targets)
X_tf = X_tf.astype(np.float32, copy=False)
X_targets = X_targets.astype(np.float32, copy=False)

print(f"Cells used: {len(cell_indices)}")
print(f"TF predictors: {len(regulators)}")
print(f"Target genes: {len(target_genes)}")

edge_rows = []
for target_col, target in enumerate(target_genes):
    # Exclude self-prediction when the target itself is one of the TF predictors.
    predictor_cols = [i for i, gene in enumerate(regulators) if gene != target]
    if len(predictor_cols) < 2:
        continue
    model = ExtraTreesRegressor(
        n_estimators=N_TREES,
        max_features=1.0,
        min_samples_leaf=2,
        random_state=SEED,
        n_jobs=-1
    )
    y = X_targets[:, target_col]
    if np.std(y) < 1e-8:
        continue
    model.fit(X_tf[:, predictor_cols], y)
    importances = model.feature_importances_
    ranked = np.argsort(importances)[::-1][:TOP_REGULATORS_PER_TARGET]
    for rank, local_j in enumerate(ranked, start=1):
        weight = float(importances[local_j])
        if weight <= MIN_EDGE_IMPORTANCE:
            continue
        regulator = regulators[predictor_cols[local_j]]
        edge_rows.append({
            "source": regulator,
            "target": target,
            "weight": weight,
            "rank_within_target": rank,
            "target_expression_variance": float(np.var(y)),
            "inference_method": "ExtraTrees feature importance"
        })

edges = pd.DataFrame(edge_rows)
if edges.empty:
    raise RuntimeError("No candidate edges were inferred. Check expression values, gene identifiers and filter settings.")
edges = edges.sort_values(["weight", "target"], ascending=[False, True]).reset_index(drop=True)
edge_path = RESULTS_DIR / "candidate_grn_edges_GSE211036.csv"
edges.to_csv(edge_path, index=False)
print("Candidate edges:", len(edges))
print("Saved network edge list:", edge_path.resolve())
display(edges.head(20))

## 8. Visualize and export the network

Edges point from candidate regulator to candidate target. The visualization displays a manageable high-importance subnetwork; the CSV contains the full set of retained ranked edges.

In [ ]:
# Build the directed graph from the strongest candidate associations.
MAX_EDGES_TO_PLOT = 160
plot_edges = edges.head(MAX_EDGES_TO_PLOT).copy()
G = nx.DiGraph()
for row in plot_edges.itertuples(index=False):
    G.add_edge(row.source, row.target, weight=row.weight)

# Keep the figure legible by displaying the top connected nodes if the graph is large.
if G.number_of_nodes() > 45:
    node_score = {}
    for u, v, data in G.edges(data=True):
        node_score[u] = node_score.get(u, 0.0) + data.get("weight", 0.0)
        node_score[v] = node_score.get(v, 0.0) + data.get("weight", 0.0)
    keep_nodes = set(sorted(node_score, key=node_score.get, reverse=True)[:45])
    G_plot = G.subgraph(keep_nodes).copy()
else:
    G_plot = G

plt.figure(figsize=(14, 10))
pos = nx.spring_layout(G_plot, seed=SEED, k=0.8 / np.sqrt(max(1, G_plot.number_of_nodes())))
node_sizes = [250 + 60 * (G_plot.in_degree(n) + G_plot.out_degree(n)) for n in G_plot.nodes()]
edge_widths = [0.5 + 4.0 * G_plot[u][v].get("weight", 0.0) for u, v in G_plot.edges()]
nx.draw_networkx_nodes(G_plot, pos, node_size=node_sizes, alpha=0.9)
nx.draw_networkx_edges(G_plot, pos, width=edge_widths, alpha=0.35, arrows=True, arrowsize=10, connectionstyle="arc3,rad=0.05")
nx.draw_networkx_labels(G_plot, pos, font_size=8)
plt.title("Candidate TF→target network: GSE211036 (top-ranked associations)")
plt.axis("off")
plt.tight_layout()
network_png = RESULTS_DIR / "candidate_grn_GSE211036.png"
plt.savefig(network_png, dpi=220, bbox_inches="tight")
plt.show()
print("Saved network figure:", network_png.resolve())

# Rank candidate regulator hubs by weighted outgoing importance.
hub_table = (edges.groupby("source", as_index=False)
             .agg(outgoing_edges=("target", "count"), total_importance=("weight", "sum"),
                  mean_importance=("weight", "mean"))
             .sort_values("total_importance", ascending=False))
hub_path = RESULTS_DIR / "candidate_regulator_hubs_GSE211036.csv"
hub_table.to_csv(hub_path, index=False)
display(hub_table.head(20))
print("Saved regulator summary:", hub_path.resolve())

## 9. Summarize regulator expression across inferred clusters

This section reports average expression of candidate TFs per computational cluster. These values are an expression summary, **not** a validated TF-activity measurement.

In [ ]:
cluster_col = "grn_cluster"
if cluster_col in adata.obs.columns:
    cluster_labels = adata.obs[cluster_col].astype(str)
    tf_present = [g for g in regulators if g in adata.var_names]
    tf_expression = adata[:, tf_present].X
    tf_expression = tf_expression.toarray() if sp.issparse(tf_expression) else np.asarray(tf_expression)
    expr_df = pd.DataFrame(tf_expression, index=adata.obs_names, columns=tf_present)
    expr_df[cluster_col] = cluster_labels.to_numpy()
    cluster_tf = expr_df.groupby(cluster_col, observed=False)[tf_present].mean().T
    cluster_tf_path = RESULTS_DIR / "TF_mean_expression_by_cluster_GSE211036.csv"
    cluster_tf.to_csv(cluster_tf_path)
    display(cluster_tf.head(20))
    print("Saved cluster TF expression:", cluster_tf_path.resolve())

    # Plot a compact heatmap for the most variable candidate TFs.
    tf_var = expr_df[tf_present].var(axis=0).sort_values(ascending=False)
    selected_tf = list(tf_var.head(min(25, len(tf_var))).index)
    plot_mat = cluster_tf.loc[selected_tf]
    plt.figure(figsize=(max(6, plot_mat.shape[1] * 1.0), max(5, plot_mat.shape[0] * 0.28)))
    plt.imshow(plot_mat.to_numpy(), aspect="auto", interpolation="nearest")
    plt.yticks(range(len(plot_mat.index)), plot_mat.index, fontsize=8)
    plt.xticks(range(len(plot_mat.columns)), plot_mat.columns, rotation=45, ha="right")
    plt.colorbar(label="Mean log-normalized expression")
    plt.title("Candidate TF expression across computational clusters")
    plt.tight_layout()
    tf_heatmap = RESULTS_DIR / "TF_expression_by_cluster_GSE211036.png"
    plt.savefig(tf_heatmap, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved heatmap:", tf_heatmap.resolve())
else:
    print("No cluster annotation found; skipped cluster-level summary.")

## 10. External benchmark datasets and immune-cell comparison

The primary GSE211036 dataset has only AML LSC/blast populations from a CEBPA-double-mutant case. To study leukemic **versus normal hematopoietic/immune** regulatory patterns, repeat the pipeline on an independent processed expression matrix from **GSE139369** and compare networks. The *Nature Biotechnology* paper integrates transcriptome, chromatin accessibility and surface-protein information in healthy hematopoiesis and MPAL; its multi-omic links can support follow-up validation of expression-derived associations.

### Dataset references
- **GSE211036** — AML LSC/blast scRNA-seq, CEBPA double-mutant AML. [NCBI GEO](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE211036); Adamo et al., [*Leukemia* (Nature Portfolio)](https://www.nature.com/articles/s41375-022-01744-5).
- **GSE139369** — healthy hematopoietic samples and mixed-phenotype acute leukemia, with scRNA-seq/scATAC-seq/CITE-seq-based regulatory analysis. [NCBI GEO](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE139369); Granja et al., [*Nature Biotechnology* (2019)](https://www.nature.com/articles/s41587-019-0332-7). The paper reports 91,601 putative peak-to-gene linkages and examples of RUNX1-linked regulatory elements near CD69.
- **GSE199336** — AML cellular hierarchy and drug-response context. [NCBI GEO](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE199336); Zeng et al., [*Nature Medicine* (2022)](https://www.nature.com/articles/s41591-022-01819-x).

### Optional processed AnnData input

Place a processed `.h5ad` file derived from GSE139369 (or another compatible immune-cell benchmark) under `data/external/`, then load it below. Ensure it has human gene symbols and document whether values are raw counts or log-normalized. Re-run the QC/preprocessing and inference sections for that dataset. Do **not** directly compare raw feature-importance values across datasets without accounting for sample size, cell composition, expression scale and model settings.

In [ ]:
# Optional: load a processed, user-prepared external dataset (e.g. GSE139369 converted to AnnData).
# This cell is deliberately opt-in: the GEO transcriptome files for GSE139369 are RDS-format,
# so the raw archive is not guaranteed to be directly readable by Scanpy.
external_h5ads = sorted(EXTERNAL_DIR.glob("*.h5ad"))
if external_h5ads:
    print("Compatible external files found:")
    for i, p in enumerate(external_h5ads, 1):
        print(f"{i}. {p}")
    EXTERNAL_H5AD = external_h5ads[0]
    adata_external = sc.read_h5ad(EXTERNAL_H5AD)
    print("Loaded external dataset:", EXTERNAL_H5AD.name, adata_external)
    print("Gene symbols example:", list(adata_external.var_names[:10]))
else:
    adata_external = None
    print("No external .h5ad found. Add a processed AnnData file to", EXTERNAL_DIR.resolve())

## 11. Interpretation checklist and recommended next steps

Before interpreting the graph biologically:

1. **Confirm identity mapping:** verify that feature names are approved HGNC symbols, not Ensembl IDs or transcript IDs.
2. **Replicate at patient level:** infer networks independently in multiple AML samples and normal/immune samples; compare edge reproducibility, not only edge weight.
3. **Control confounding:** account for cell-cycle effects, library size, batch, patient and cell-state composition. Correlation/feature importance may be driven by shared cell state.
4. **Validate edges:** prioritize candidate TFs and targets supported by TF motifs in accessible peaks, ChIP-seq/CUT&Tag, peak-to-gene links, perturbation, and the published regulatory resources.
5. **Distinguish TF expression from TF activity:** mRNA abundance is not a direct readout of protein activity.
6. **Use suitable methods for a formal benchmark:** compare with GENIE3, GRNBoost2/SCENIC, CellOracle or another method against a documented gold standard; use a versioned TF catalogue and report hyperparameters.

### Outputs
- `results/candidate_grn_edges_GSE211036.csv` — inferred regulator→target edge list with importance scores.
- `results/candidate_regulator_hubs_GSE211036.csv` — regulator-level summary.
- `results/candidate_grn_GSE211036.png` — network visualization.
- `results/TF_mean_expression_by_cluster_GSE211036.csv` and `results/TF_expression_by_cluster_GSE211036.png` — cluster expression summaries.
- `results/GSE211036_preprocessed.h5ad` — processed AnnData object.

**Not clinical advice or a clinical diagnostic model.** This notebook is a research starting point; results require independent biological validation.